# 02 · Análisis descriptivo y visualización
Usamos el dataset limpio del notebook 01. La **tasa de conversión** es el % de llamadas en las que el cliente suscribió el depósito (`y = 1`).

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.funciones import tasa_conversion

sns.set_theme(style="whitegrid")
df = pd.read_csv("../data/processed/bank_marketing_clean.csv")
print(df.shape)

## 1. Estadística descriptiva

In [ ]:
numericas = ["age", "duration", "campaign", "previous", "Income", "NumWebVisitsMonth",
             "antiguedad_anios", "euribor3m", "emp.var.rate", "cons.price.idx",
             "cons.conf.idx", "nr.employed"]

resumen = df[numericas].describe().T[["count", "mean", "50%", "std", "min", "max"]]
resumen = resumen.rename(columns={"50%": "mediana"})
resumen["moda"] = df[numericas].mode().iloc[0]
resumen = resumen.round(2)
resumen.to_csv("../reports/tables/descriptivo_numericas.csv")
resumen

In [ ]:
tasa_global = df["y"].mean() * 100
print(f"Tasa de conversión global: {tasa_global:.2f}% ({df['y'].sum()} de {len(df)} llamadas)")

In [ ]:
# Distribución (%) de las variables categóricas
for col in ["job", "marital", "education", "contact", "poutcome"]:
    print(col)
    print((df[col].value_counts(normalize=True) * 100).round(1))
    print()

**Observación:** `duration` y `campaign` tienen la media muy por encima de la mediana (hay llamadas muy largas y clientes muy insistidos),
así que para describir al cliente típico es mejor la mediana.

## 2. Distribuciones

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
for eje, (col, titulo) in zip(ejes, [("age", "Edad"), ("Income", "Ingresos anuales"), ("duration", "Duración de la llamada (s)")]):
    sns.histplot(df[col], bins=40, kde=True, ax=eje, color="steelblue")
    eje.axvline(df[col].median(), color="red", linestyle="--", label=f"Mediana: {df[col].median():.0f}")
    eje.set_title(titulo)
    eje.set_xlabel("")
    eje.legend()
plt.tight_layout()
plt.savefig("../reports/figures/01_distribuciones.png", dpi=120)
plt.show()

## 3. Tasa de conversión por segmentos
Función auxiliar para no repetir el mismo código de gráfico varias veces.

In [ ]:
def grafico_barras(tabla, titulo, archivo, horizontal=False):
    """Dibuja la tasa de conversión de una tabla (índice = categorías) y la guarda."""
    plt.figure(figsize=(8, 5) if horizontal else (7, 4))
    if horizontal:
        ax = sns.barplot(x=tabla["tasa_%"], y=tabla.index.astype(str), color="steelblue")
        ax.axvline(tasa_global, color="red", linestyle="--")
    else:
        ax = sns.barplot(x=tabla.index.astype(str), y=tabla["tasa_%"], color="steelblue")
        ax.axhline(tasa_global, color="red", linestyle="--")
    ax.bar_label(ax.containers[0], fmt="%.1f%%")
    ax.set_title(titulo + "  (línea roja = media global)")
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.savefig(f"../reports/figures/{archivo}.png", dpi=120)
    plt.show()

In [ ]:
por_trabajo = tasa_conversion(df, "job")
grafico_barras(por_trabajo, "Conversión por profesión (%)", "02_profesion", horizontal=True)
por_trabajo

In [ ]:
por_edad = tasa_conversion(df, "grupo_edad").sort_index()      # sort_index = orden natural de los grupos
grafico_barras(por_edad, "Conversión por grupo de edad (%)", "03_edad")
por_edad

In [ ]:
# ¿La edad influye solo porque hay muchos jubilados y estudiantes? Los quitamos y comprobamos
sin_jub_est = df[~df["job"].isin(["retired", "student"])]
tasa_conversion(sin_jub_est, "grupo_edad").sort_index()

In [ ]:
por_poutcome = tasa_conversion(df, "poutcome")
grafico_barras(por_poutcome, "Conversión según resultado de la campaña anterior (%)", "04_poutcome")
por_poutcome

In [ ]:
por_canal = tasa_conversion(df, "contact")
grafico_barras(por_canal, "Conversión por canal de contacto (%)", "05_canal")
por_canal

## 4. Número de contactos y duración de la llamada

In [ ]:
df["tramo_campaign"] = pd.cut(df["campaign"], bins=[0, 1, 2, 3, 5, 10, 100],
                              labels=["1", "2", "3", "4-5", "6-10", "11+"])
por_campaign = tasa_conversion(df, "tramo_campaign").sort_index()
grafico_barras(por_campaign, "Conversión según nº de contactos (%)", "06_campaign")
por_campaign

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="y", y="duration", showfliers=False, color="lightsteelblue")
plt.xticks([0, 1], ["No suscribe", "Suscribe"])
plt.title("Duración de la llamada según resultado")
plt.xlabel("")
plt.ylabel("Segundos")
plt.tight_layout()
plt.savefig("../reports/figures/07_duracion.png", dpi=120)
plt.show()

df.groupby("y")["duration"].describe().round(0)

Quien suscribe habla bastante más tiempo. **Cuidado:** la duración solo se conoce *después* de la llamada, así que no sirve para decidir
a quién llamar. Es una consecuencia del interés del cliente, no una causa.

## 5. Contexto macroeconómico

In [ ]:
# Dividimos el euribor en 5 tramos con el mismo nº de clientes
df["tramo_euribor"] = pd.qcut(df["euribor3m"], q=5, duplicates="drop")
por_euribor = tasa_conversion(df, "tramo_euribor").sort_index()
grafico_barras(por_euribor, "Conversión según el euribor a 3 meses (%)", "08_euribor")
por_euribor

In [ ]:
columnas_corr = ["y", "age", "duration", "campaign", "previous", "Income", "NumWebVisitsMonth",
                 "hijos_total", "antiguedad_anios", "emp.var.rate", "cons.price.idx",
                 "cons.conf.idx", "euribor3m", "nr.employed"]
correlaciones = df[columnas_corr].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlaciones, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7})
plt.title("Matriz de correlación")
plt.tight_layout()
plt.savefig("../reports/figures/09_correlaciones.png", dpi=120)
plt.show()

correlaciones["y"].drop("y").sort_values().round(3)

`euribor3m`, `emp.var.rate` y `nr.employed` están muy correladas entre sí (≈ 0,9): miden casi lo mismo.

## 6. Variables del Excel: ingresos, hijos y visitas web

In [ ]:
df["tramo_web"] = pd.qcut(df["NumWebVisitsMonth"], q=4)
for col in ["tramo_ingresos", "hijos_total", "tramo_web"]:
    print(tasa_conversion(df, col).sort_index(), "\n")

Las tasas son casi idénticas en todos los grupos: **ingresos, hijos y visitas web no influyen**.

## 7. Año de alta del cliente (hoja del Excel)

In [ ]:
por_cohorte = tasa_conversion(df, "anio_hoja").sort_index()
grafico_barras(por_cohorte, "Conversión según año de alta (%)", "10_anio_alta")
por_cohorte

In [ ]:
# ¿Es un efecto real? Comparamos solo clientes sin campaña previa y con euribor alto (> 4)
control = df[(df["poutcome"] == "nonexistent") & (df["euribor3m"] > 4)]
tasa_conversion(control, "anio_hoja").sort_index()

En el gráfico anterior los clientes de 2014 parecen convertir mucho más, pero al compararlos en igualdad de condiciones **la diferencia desaparece**.
La hoja del Excel sigue el orden temporal del dataset original, así que se mezcla con el ciclo económico (euribor bajo) y con el contacto previo.
Es un ejemplo de que **correlación no es causalidad**.

## 8. Fecha de la llamada

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
por_anio = df.groupby("contact_year")["y"].mean() * 100
por_mes = df.groupby("contact_month")["y"].mean() * 100
sns.barplot(x=por_anio.index.astype(int), y=por_anio.values, color="steelblue", ax=ejes[0])
sns.barplot(x=por_mes.index.astype(int), y=por_mes.values, color="steelblue", ax=ejes[1])
ejes[0].set_title("Conversión por año de la llamada (%)")
ejes[1].set_title("Conversión por mes de la llamada (%)")
for eje in ejes:
    eje.axhline(tasa_global, color="red", linestyle="--")
    eje.set_xlabel("")
plt.tight_layout()
plt.savefig("../reports/figures/11_fecha_llamada.png", dpi=120)
plt.show()
print(por_anio.round(2).to_dict())

No hay estacionalidad ni tendencia clara: la conversión se mueve siempre entre el 10% y el 12%.